<a href="https://colab.research.google.com/github/ericksbr/2026-data-short-course/blob/main/notebooks/noaa_baseline1_mcdonough_mckenzie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericksbr/noaa_comments_dedup/blob/main/notebooks/noaa_baseline1_mcdonough_mckenzie.ipynb)

**Note:** This notebook is in a private repository. Opening in Colab requires a GitHub account with access to this repo.

<a href="https://colab.research.google.com/github/ericksbr/noaa_comments_dedup/blob/main/notebooks/noaa_baseline1_mcdonough_mckenzie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Baseline 1: McDonough-McKenzie et al. (2020)

## Study 2 Deduplication Pipeline Comparison

This notebook creates **Baseline 1**, which replicates the deduplication method from McDonough-McKenzie et al. (2020) across all 8 selected NOAA dockets,  following the exact implementation verified from the published repository  (`gitlab.com/tonychang/smith-monuments-sentiment`, notebooks `06-Filter_Comments.ipynb` and `10-Cosine_similarity.ipynb`).

**Two-stage pipeline:**

1. **Exact-text deduplication**: Groups comments by exact byte-for-byte text, keeps first occurrence.
2. **TF-IDF cosine-similarity near-duplicate detection**: Uses `TfidfVectorizer ()` with all defaults, computes pairwise cosine similarities in row-batches (following the authors' 512-bin `linspace` approach), and marks comments whose maximum similarity to any other comment exceeds a threshold as near-duplicates.

**Implementation notes:**

- **Threshold**: The paper text states 0.90 but reports testing multiple values; the published repo's actual code uses 0.95. We report both plus 0.80/0.85 for completeness.
- **Memory efficiency**: For dockets with ~100k comments, we cannot store a full `n × n` dense cosine-similarity matrix in memory. We follow the published authors' approach: partition rows into 512 bins via `np.linspace(0, n, 512)`, compute `cosine_similarity(tfmat[bin_start:bin_end], tfmat)` per bin, extract per-row max, then discard the dense block before the next bin.
- **Output**: Combined summary table (one row per docket × threshold) and per-docket row-level flags, saved to `data/study2/baseline1_results/`.

**Note:** When initially run (8/17/2026), text had not been extracted from docket attachments.

## 0.1 — Install Dependencies

In [1]:
!pip install -q pandas numpy scikit-learn pyarrow psutil

## 0.2 — Mount Google Drive & Set Paths

In [2]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

# Corresponds to G:\My Drive\claude_code\noaa-comments-dedup on a machine with Google Drive for Desktop
PROJECT_ROOT = '/content/drive/MyDrive/claude_code/noaa-comments-dedup'
DOCKETS_DIR = os.path.join(PROJECT_ROOT, 'data', 'study2', 'dockets')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'data', 'study2', 'baseline1_results')

# Create results directory if it doesn't exist
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Dockets dir: {DOCKETS_DIR}")
print(f"✓ Results dir: {RESULTS_DIR}")

Mounted at /content/drive
✓ Project root: /content/drive/MyDrive/claude_code/noaa-comments-dedup
✓ Dockets dir: /content/drive/MyDrive/claude_code/noaa-comments-dedup/data/study2/dockets
✓ Results dir: /content/drive/MyDrive/claude_code/noaa-comments-dedup/data/study2/baseline1_results


## 0.3 — Import Libraries

In [3]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import warnings
import time
import psutil
import os

warnings.filterwarnings('ignore')
print('✓ Dependencies imported.')

✓ Dependencies imported.


## 1.0 — Load Selected Dockets

In [4]:
# Your 8 selected dockets
selected_dockets = [
    'NOAA-NMFS-2019-0045',
    'NOAA-NMFS-2019-0073',
    'NOAA-NMFS-2021-0075',
    'NOAA-NOS-2017-0066',
    'NOAA-NMFS-2023-0089',
    'NOAA-NMFS-2022-0119',
    'NOAA-NMFS-2023-0117',
    'NOAA-NMFS-2023-0027',
]

# Verify files exist
print(f"Checking docket files in {DOCKETS_DIR}...\n")
docket_files = {}

for docket_id in selected_dockets:
    parquet_path = os.path.join(DOCKETS_DIR, docket_id, f'{docket_id}.parquet')
    if os.path.exists(parquet_path):
        docket_files[docket_id] = parquet_path
        print(f"  ✓ {docket_id}")
    else:
        print(f"  ✗ {docket_id} — FILE NOT FOUND")

print(f"\n✓ Found {len(docket_files)}/{len(selected_dockets)} docket files")

Checking docket files in /content/drive/MyDrive/claude_code/noaa-comments-dedup/data/study2/dockets...

  ✓ NOAA-NMFS-2019-0045
  ✓ NOAA-NMFS-2019-0073
  ✓ NOAA-NMFS-2021-0075
  ✓ NOAA-NOS-2017-0066
  ✓ NOAA-NMFS-2023-0089
  ✓ NOAA-NMFS-2022-0119
  ✓ NOAA-NMFS-2023-0117
  ✓ NOAA-NMFS-2023-0027

✓ Found 8/8 docket files


## 2.0 — Define Baseline 1 Pipeline Function

In [6]:
def run_baseline1(docket_id, parquet_path, comment_col='Comment', thresholds=None):
    """
    Run Baseline 1 (McDonough-McKenzie et al. 2020) on a single docket.
    Includes cost tracking (runtime, memory) for Study 2 outcome measures.

    Args:
        docket_id (str): Docket identifier
        parquet_path (str): Path to docket parquet file
        comment_col (str): Name of comment text column
        thresholds (list): Cosine similarity thresholds to test (default: [0.80, 0.85, 0.90, 0.95])

    Returns:
        dict: Results including exact-dup stats, max similarities, threshold-sweep classifications, and cost metrics
    """

    if thresholds is None:
        thresholds = [0.80, 0.85, 0.90, 0.95]

    # Get process for memory tracking
    process = psutil.Process(os.getpid())

    # Overall start time
    t_start_total = time.perf_counter()
    mem_start = process.memory_info().rss / 1024 / 1024  # MB

    # Load data
    t_start = time.perf_counter()
    df = pd.read_parquet(parquet_path)
    t_load = time.perf_counter() - t_start

    # Filter out null/empty comments
    df_work = df[df[comment_col].notna() & (df[comment_col].str.strip() != '')].copy()
    n_dropped = len(df) - len(df_work)

    # ========================================================================
    # STAGE A: Exact-text deduplication
    # ========================================================================

    t_start = time.perf_counter()
    text_counts = df_work[comment_col].value_counts()
    n_exact_groups = len(text_counts)
    n_exact_duplicates = len(df_work) - n_exact_groups

    # Keep only first occurrence of each unique text
    df_exact_dedup = df_work.drop_duplicates(subset=[comment_col], keep='first')
    t_stage_a = time.perf_counter() - t_start
    mem_after_stage_a = process.memory_info().rss / 1024 / 1024  # MB

    # ========================================================================
    # STAGE B: TF-IDF + binned cosine similarity (following GitLab source)
    # ========================================================================

    t_start = time.perf_counter()
    texts = df_exact_dedup[comment_col].tolist()
    n = len(texts)

    # TF-IDF vectorization (all defaults, matching published repo)
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
    t_vectorize = time.perf_counter() - t_start
    mem_after_vectorize = process.memory_info().rss / 1024 / 1024  # MB

    # Create 512 bins using linspace (published authors' approach)
    z = np.linspace(0, n, 512).astype(int)

    # Preallocate max similarity array
    max_sim_per_comment = np.zeros(n)

    # Process each bin
    t_start = time.perf_counter()
    for i in range(1, len(z)):
        bin_start = z[i-1]
        bin_end = z[i]

        # Compute cosine similarity: bin rows vs all rows
        c = cosine_similarity(tfidf_matrix[bin_start:bin_end], tfidf_matrix)

        # Zero out self-similarity (exact matches: diagonal = 1)
        c[c == 1] = 0

        # Extract per-row max and store
        max_sim_per_comment[bin_start:bin_end] = c.max(axis=1)

        # Discard dense block (memory efficiency)
        del c

    t_cosine = time.perf_counter() - t_start
    mem_after_cosine = process.memory_info().rss / 1024 / 1024  # MB

    # ========================================================================
    # Threshold sweep
    # ========================================================================

    t_start = time.perf_counter()
    baseline1_results = {}

    for threshold in thresholds:
        is_duplicate = max_sim_per_comment > threshold
        n_duplicates = is_duplicate.sum()
        n_unique = (~is_duplicate).sum()
        pct_dup = 100 * n_duplicates / len(is_duplicate) if len(is_duplicate) > 0 else 0

        baseline1_results[threshold] = {
            'is_duplicate': is_duplicate,
            'max_sims': max_sim_per_comment,
            'n_duplicate': n_duplicates,
            'n_unique': n_unique,
            'pct_dup': pct_dup
        }

    t_threshold = time.perf_counter() - t_start

    # Final memory and total time
    t_total = time.perf_counter() - t_start_total
    mem_end = process.memory_info().rss / 1024 / 1024  # MB
    mem_peak = mem_end  # Rough peak (not exact, but captures general usage)

    # Return results
    return {
        'docket_id': docket_id,
        'total_rows': len(df_work),
        'n_exact_duplicates': n_exact_duplicates,
        'n_after_exact_dedup': len(df_exact_dedup),
        'max_sims': max_sim_per_comment,
        'threshold_results': baseline1_results,
        'df_exact_dedup': df_exact_dedup,  # Keep for later joins
        # Cost metrics
        'cost': {
            'time_load_sec': t_load,
            'time_stage_a_sec': t_stage_a,
            'time_vectorize_sec': t_vectorize,
            'time_cosine_sec': t_cosine,
            'time_threshold_sec': t_threshold,
            'time_total_sec': t_total,
            'mem_start_mb': mem_start,
            'mem_after_stage_a_mb': mem_after_stage_a,
            'mem_after_vectorize_mb': mem_after_vectorize,
            'mem_after_cosine_mb': mem_after_cosine,
            'mem_end_mb': mem_end,
            'mem_peak_mb': mem_peak,
            }
        }

## 3.0 — Run Baseline 1 Across All 8 Dockets

In [7]:
all_results = []
summary_rows = []

thresholds = [0.80, 0.85, 0.90, 0.95]

print("="*80)
print("BASELINE 1: McDonough-McKenzie et al. (2020) — Running Pipeline")
print("="*80)

for docket_id in selected_dockets:
    if docket_id not in docket_files:
        print(f"\n⚠ Skipping {docket_id}: file not found")
        continue

    print(f"\n{'='*80}")
    print(f"DOCKET: {docket_id}")
    print(f"{'='*80}")

    # Run pipeline
    result = run_baseline1(docket_id, docket_files[docket_id], thresholds=thresholds)
    all_results.append(result)

    # Print Stage A results
    print(f"\nStage A: Exact-text deduplication")
    print(f"  Total comments: {result['total_rows']:,}")
    print(f"  Exact duplicates removed: {result['n_exact_duplicates']:,}")
    print(f"  Unique texts: {result['n_after_exact_dedup']:,}")

    # Print Stage B results (threshold sweep)
    print(f"\nStage B: TF-IDF cosine-similarity near-duplicate detection")
    print(f"  Classification results:")

    for threshold in thresholds:
        res = result['threshold_results'][threshold]
        print(f"\n  Threshold: {threshold}")
        print(f"    Near-duplicates (max_sim > {threshold}): {res['n_duplicate']:,} ({res['pct_dup']:.1f}%)")
        print(f"    Unique/genuine: {res['n_unique']:,} ({100-res['pct_dup']:.1f}%)")

        # Store for summary table
        summary_rows.append({
            'docket_id': docket_id,
            'total_comments': result['total_rows'],
            'exact_duplicates': result['n_exact_duplicates'],
            'threshold': threshold,
            'n_near_duplicate': res['n_duplicate'],
            'n_unique': res['n_unique'],
            'pct_dup': res['pct_dup']
        })

print(f"\n{'='*80}")
print(f"✓ Completed {len(all_results)} dockets")
print(f"{'='*80}")

BASELINE 1: McDonough-McKenzie et al. (2020) — Running Pipeline

DOCKET: NOAA-NMFS-2019-0045

Stage A: Exact-text deduplication
  Total comments: 594
  Exact duplicates removed: 2
  Unique texts: 592

Stage B: TF-IDF cosine-similarity near-duplicate detection
  Classification results:

  Threshold: 0.8
    Near-duplicates (max_sim > 0.8): 348 (58.8%)
    Unique/genuine: 244 (41.2%)

  Threshold: 0.85
    Near-duplicates (max_sim > 0.85): 346 (58.4%)
    Unique/genuine: 246 (41.6%)

  Threshold: 0.9
    Near-duplicates (max_sim > 0.9): 346 (58.4%)
    Unique/genuine: 246 (41.6%)

  Threshold: 0.95
    Near-duplicates (max_sim > 0.95): 337 (56.9%)
    Unique/genuine: 255 (43.1%)

DOCKET: NOAA-NMFS-2019-0073

Stage A: Exact-text deduplication
  Total comments: 6,639
  Exact duplicates removed: 4,750
  Unique texts: 1,889

Stage B: TF-IDF cosine-similarity near-duplicate detection
  Classification results:

  Threshold: 0.8
    Near-duplicates (max_sim > 0.8): 1,617 (85.6%)
    Unique/genu

KeyboardInterrupt: 

## 4.0 — Save Results

In [ ]:
# Create combined summary table
summary_df = pd.DataFrame(summary_rows)

# Save summary to CSV and parquet
summary_csv_path = os.path.join(RESULTS_DIR, 'baseline1_summary.csv')
summary_parquet_path = os.path.join(RESULTS_DIR, 'baseline1_summary.parquet')

summary_df.to_csv(summary_csv_path, index=False)
summary_df.to_parquet(summary_parquet_path, index=False)

print(f"✓ Saved summary table:")
print(f"  {summary_csv_path}")
print(f"  {summary_parquet_path}")
print(f"\n  Dimensions: {len(summary_df)} rows × {len(summary_df.columns)} columns")

print(f"\nSummary table:")
print(summary_df.to_string(index=False))

## 4.1 — Save Per-Docket Row-Level Flags

In [ ]:
# Save per-docket row-level near-duplicate flags for later joining with raw data
print("Saving per-docket row-level flags...\n")

for result in all_results:
    docket_id = result['docket_id']

    # Create output directory
    docket_result_dir = os.path.join(RESULTS_DIR, docket_id)
    os.makedirs(docket_result_dir, exist_ok=True)

    # Build row-level dataframe with Document ID from the deduped data
    df_flags = result['df_exact_dedup'][['Document ID']].copy()
    df_flags['max_sim'] = result['max_sims']

    # Add per-threshold classification flags
    for threshold in thresholds:
        col_name = f'is_near_dup_th{threshold}'
        df_flags[col_name] = result['threshold_results'][threshold]['is_duplicate']

    # Save
    flags_parquet = os.path.join(docket_result_dir, f'{docket_id}_baseline1_flags.parquet')
    df_flags.to_parquet(flags_parquet, index=False)

    print(f"  ✓ {docket_id}: {len(df_flags):,} rows saved to {flags_parquet}")

print(f"\n✓ All per-docket flags saved")

## 5.0 — Summary Statistics

In [ ]:
print("\n" + "="*80)
print("BASELINE 1: McDonough-McKenzie et al. (2020) — Final Summary")
print("="*80)

for result in all_results:
    docket_id = result['docket_id']
    print(f"\n{docket_id}")
    print(f"  Original rows: {result['total_rows']:,}")
    print(f"  Exact duplicates removed: {result['n_exact_duplicates']:,}")
    print(f"  After exact dedup: {result['n_after_exact_dedup']:,}")

    for threshold in thresholds:
        res = result['threshold_results'][threshold]
        print(f"    Threshold {threshold}: near-dup={res['n_duplicate']:>5,} ({res['pct_dup']:>5.1f}%), unique={res['n_unique']:>5,} ({100-res['pct_dup']:>5.1f}%)")

print(f"\n{'='*80}")
print(f"✓ Results saved to: {RESULTS_DIR}")
print(f"  - baseline1_summary.csv / .parquet")
print(f"  - {{docket_id}}/{{docket_id}}_baseline1_flags.parquet (per-docket row-level flags)")
print(f"{'='*80}")

## 10.0 — Grouping Comparison: Document-level vs. Sentence-level (Placeholder)

**Status:** Placeholder; Baseline 1 is document-only. Sentence-level variant will be implemented in the modular pipeline (Study 2).

**Plan (when implemented):**
- Apply a sentence-level clustering variant: embed sentences separately, cluster sentences, then aggregate sentence-cluster membership back to documents.
- Compare document-level (Baseline 1) vs. sentence-level grouping: do they agree on which comments belong to the same campaign/form-letter?
- Motivation: Long insertions can push template text past embedding model token limits; sentence-level avoids this and may recover campaign structure better.
- Test especially on comments with large insertion/truncation/modification patterns.

**Output format (when ready):**
- `baseline1_vs_sentence_level.csv`: One row per (docket_id × threshold), columns = `docket_id, threshold, doc_level_clusters, sentence_level_clusters, adjusted_rand_index, recovery_pct, ...`

## 9.0 — Stability Metrics (Placeholder)

**Status:** Requires multiple baselines for comparison (Baseline 2, Baseline 3, modular pipeline).

**Plan (when implemented):**
- For each comment, track whether Baseline 1 (at different thresholds) agrees with other methods on the duplicate/modified/unique category.
- Compute pairwise agreement (Cohen's kappa, Krippendorff's alpha) between Baseline 1 and each other method, per docket.
- Identify disagreement patterns: which comments are classified inconsistently across methods? Are they near the decision boundary in Baseline 1?

**Output format (when ready):**
- `baseline1_stability_vs_baselines.csv`: One row per (docket_id × threshold × other_method), columns = `docket_id, threshold, other_method, agreement_pct, kappa, ...`
- `baseline1_boundary_comments.parquet`: Row-level, identifying comments where Baseline 1's classification is near threshold or disagrees with other methods.

## 8.0 — Cluster Quality Without Labels (Placeholder)

**Status:** Unsupervised diagnostic metrics; can be computed now.

**Plan (when implemented):**
- **Silhouette score:** Treat each threshold's near-duplicate grouping as a clustering, compute silhouette coefficient using cosine distance.
- **Centroid similarity:** For each near-duplicate group, compute average pairwise cosine similarity within the group; aggregate across dockets.
- **Singleton cohesion:** Comments classified as unique (not near-duplicates) should have low max-similarity (check mean + distribution).

**Output format (when ready):**
- `baseline1_cluster_quality.csv`: One row per (docket_id × threshold), columns = `docket_id, threshold, silhouette, avg_within_group_sim, pct_singleton, ...`

## 7.0 — Accuracy Metrics (Placeholder)

**Status:** Requires hand-labeled data (Study 1 practitioner ratings + additional labeled test set).

**Plan:**
- Once Study 1 ordinal logistic regression produces similarity cut points (duplicate/modified/unique boundaries), we'll apply them to Baseline 1's `max_sim_per_comment` to classify each comment into the 3-category scheme.
- Compare against hand-labeled test set using: pairwise accuracy, confusion matrix, per-category F1.
- Also test Baseline 1's classifications against Study 1's practitioner-rated item set directly.

**Output format (when ready):**
- `baseline1_accuracy.csv`: One row per docket, columns = `docket_id, num_labeled_comments, accuracy, precision_dup, recall_dup, f1_dup, ...` (for each category).

In [ ]:
# Extract and save cost metrics
cost_rows = []

for result in all_results:
    docket_id = result['docket_id']
    cost = result['cost']

    cost_rows.append({
        'docket_id': docket_id,
        'num_comments': result['n_after_exact_dedup'],
        'time_load_sec': cost['time_load_sec'],
        'time_stage_a_sec': cost['time_stage_a_sec'],
        'time_vectorize_sec': cost['time_vectorize_sec'],
        'time_cosine_sec': cost['time_cosine_sec'],
        'time_threshold_sec': cost['time_threshold_sec'],
        'time_total_sec': cost['time_total_sec'],
        'mem_start_mb': cost['mem_start_mb'],
        'mem_after_stage_a_mb': cost['mem_after_stage_a_mb'],
        'mem_after_vectorize_mb': cost['mem_after_vectorize_mb'],
        'mem_after_cosine_mb': cost['mem_after_cosine_mb'],
        'mem_end_mb': cost['mem_end_mb'],
        'mem_peak_mb': cost['mem_peak_mb'],
    })

cost_df = pd.DataFrame(cost_rows)

# Save cost metrics
cost_csv_path = os.path.join(RESULTS_DIR, 'baseline1_cost_metrics.csv')
cost_parquet_path = os.path.join(RESULTS_DIR, 'baseline1_cost_metrics.parquet')

cost_df.to_csv(cost_csv_path, index=False)
cost_df.to_parquet(cost_parquet_path, index=False)

print("Cost Metrics Summary")
print("=" * 100)
print(cost_df.to_string(index=False))
print(f"\n✓ Saved to: {cost_csv_path} / {cost_parquet_path}")

## 6.0 — Cost Metrics (Runtime & Memory)